# 模型测试
> 用于模型状态检查

## 模型定义

In [3]:
import torch
import torch.nn as nn

class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(10, 10)       
        self.layers = nn.ModuleDict({
            "0": nn.Linear(10, 10),
            "1": nn.Linear(10, 10),
        })
        self.head = nn.Linear(10, 20)              

    def forward(self, x):
        x = self.embed(x)
        for _, blk in self.layers.items():
            x = blk(x)
        return self.head(x)

tmd = TinyModel()
tmd

TinyModel(
  (embed): Embedding(10, 10)
  (layers): ModuleDict(
    (0): Linear(in_features=10, out_features=10, bias=True)
    (1): Linear(in_features=10, out_features=10, bias=True)
  )
  (head): Linear(in_features=10, out_features=20, bias=True)
)

In [4]:
for layer_id, transformer_block in tmd.layers.items():
    print(f"layer_id: {layer_id}")
    print(f"transformer_block: {transformer_block}")


layer_id: 0
transformer_block: Linear(in_features=10, out_features=10, bias=True)
layer_id: 1
transformer_block: Linear(in_features=10, out_features=10, bias=True)


In [2]:
# 打印模型参数
def print_model_params(model):
    for name, param in model.named_parameters():
        print(f"-----------{name}---------------\n "
              f"shape={tuple(param.shape)}  dtype={param.dtype}  device={param.device}\n"
              f" [Params]:\n{param}\n"
              f"------------------\n")
print_model_params(tmd)
# for name, param in tmd.named_parameters():
#     print(f"-----------{name}---------------\n "
#           f"shape={tuple(param.shape)}  dtype={param.dtype}  device={param.device}\n"
#           f" [Params]:\n{param}\n"
#           f"------------------\n")

-----------embed.weight---------------
 shape=(10, 10)  dtype=torch.float32  device=cpu
 [Params]:
Parameter containing:
tensor([[-0.4072,  0.3286,  0.6756, -0.2521, -1.7226, -0.3680,  1.6527, -1.2239,
         -0.8977, -0.6901],
        [ 0.8479,  0.7344, -0.7781,  1.7044, -0.3118,  0.0897,  0.2081,  0.3780,
         -0.0490,  0.1771],
        [ 0.5790, -0.2912,  0.9886,  0.6286, -1.8065, -0.3026, -0.2960, -2.0930,
         -0.5433, -1.4410],
        [-0.2129,  0.3240,  0.6723,  0.2511, -1.2899, -1.2450, -0.5882, -0.9440,
          0.4219,  1.1517],
        [-1.7113,  0.1433, -0.2514,  0.1423,  0.7307, -1.0617,  0.0448,  0.0426,
          1.0268, -0.3198],
        [ 1.3606, -0.1987,  0.5936,  0.6182, -0.4550,  0.2581,  0.3261,  0.8248,
         -0.6065, -0.0319],
        [-1.1300,  0.0968, -0.2599, -0.6785, -0.6636,  1.5077, -1.1816, -0.0169,
          0.5488,  1.1448],
        [ 1.6036,  0.3289,  0.1727,  0.0808, -1.5903, -0.8219, -0.5790, -0.2713,
          1.7920, -0.0999],
       

## 模拟分布式环境

In [ ]:
import os
import torch
import torch.distributed as dist
from torch.distributed.device_mesh import init_device_mesh
from torch.distributed._composable.fsdp import fully_shard, MixedPrecisionPolicy
# 设置单进程模拟环境
os.environ['RANK'] = '0'
os.environ['WORLD_SIZE'] = '2'
os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '12355'


dist.init_process_group(backend='gloo',rank=0, world_size=2) 

cpu_local_mesh = init_device_mesh("cpu", mesh_shape=(2,))

mp_policy = MixedPrecisionPolicy(
            param_dtype=torch.bfloat16
        )

In [ ]:
for layer_id, transformer_block in tmd.layers.items():
    if config.train.reshard_after_forward:
        reshard_after_forward = int(layer_id) < len(model.layers) - 1
    else:
        reshard_after_forward = False
    fully_shard(
        transformer_block,
        mp_policy=mp_policy,
        mesh=elastic_device_mesh.cuda_local_mesh,
        reshard_after_forward=reshard_after_forward,
    )
print_model_params(tmd)


In [ ]:
fully_shard(
    tmd,
    mp_policy=mp_policy,
    mesh=elastic_device_mesh.cuda_local_mesh,
    reshard_after_forward=config.train.reshard_after_forward,
)
print_model_params(tmd)


In [ ]:
dist.destroy_process_group()